# Temporal-Difference Learning

Temporal-difference (TD) learning combines ideas from Monte Carlo methods and dynamic programming (DP):

- Like Monte Carlo methods, TD methods learn directly from raw experience without a model of the environment's dynamics.
- Like DP, TD methods update estimates partly from other learned estimates, without waiting for a final outcome; they bootstrap.

We begin with the **policy evaluation**, or **prediction**, problem: estimating the value function $v_\pi$ for a given policy $\pi$. For the **control** problem of finding an optimal policy, DP, TD, and Monte Carlo methods use variations of generalized policy iteration (GPI); their primary differences are their approaches to prediction.

## TD Prediction

Both TD and Monte Carlo methods solve the prediction problem from experience generated by following a policy $\pi$, but they update their estimates differently.

A simple every-visit Monte Carlo method uses the return $G_t$ as its target after the return is known:

$$
V(S_t) \leftarrow V(S_t) + \alpha\left[G_t - V(S_t)\right],
\tag{6.1}
$$

where $G_t$ is the actual return following time $t$ and $\alpha$ is a constant step-size parameter, $0 < \alpha \leq 1$. Monte Carlo methods must wait until the end of the episode because $G_t$ is known only then.

TD methods need wait only until the next time step. At time $t+1$, they immediately form a target from the observed reward $R_{t+1}$ and the estimate $V(S_{t+1})$. The simplest TD method updates

$$
V(S_t) \leftarrow V(S_t) + \alpha\left[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)\right].
\tag{6.2}
$$

This method is called **TD(0)**, or **one-step TD**, because it uses $R_{t+1} + \gamma V(S_{t+1})$ as its target. It bootstraps because this target is based partly on the existing estimate $V(S_{t+1})$.

### Tabular TD(0) for Estimating $v_\pi$

**Input:** the policy $\pi$ to be evaluated  
**Algorithm parameter:** step size $\alpha \in (0,1]$  
**Initialize:** $V(s)$ arbitrarily for all $s \in \mathcal{S}^+$, except $V(\text{terminal}) = 0$

```text
Loop for each episode:
    Initialize S
    Loop for each step of episode:
        A <- action given by pi for S
        Take action A, observe R, S'
        V(S) <- V(S) + alpha[R + gamma V(S') - V(S)]
        S <- S'
    until S is terminal
```

### Targets and Backup

The value function satisfies

$$
v_\pi(s) \doteq \mathbb{E}_\pi[G_t \mid S_t = s].
\tag{6.3}
$$

Using the recursive definition of the return gives

$$
\begin{aligned}
v_\pi(s)
&= \mathbb{E}_\pi[R_{t+1} + \gamma G_{t+1} \mid S_t = s] \\
&= \mathbb{E}_\pi[R_{t+1} + \gamma v_\pi(S_{t+1}) \mid S_t = s].
\end{aligned}
\tag{6.4}
$$

Monte Carlo methods use the sample return $G_t$ as an estimate of (6.3). DP methods use an estimate of (6.4): the expected values are provided by the model, but $v_\pi(S_{t+1})$ is unknown and the current estimate $V(S_{t+1})$ is used instead. The TD target estimates (6.4) for both reasons: it samples the expected values and uses the current estimate $V$ in place of the true $v_\pi$.

The TD(0) backup uses the value estimate for $S_t$ and one sample transition to the immediately following state. The update is based on this single sample successor rather than a complete distribution of possible successors. TD methods combine the sampling of Monte Carlo methods with the bootstrapping of DP methods.

### TD Error

The quantity in brackets in the TD(0) update is the **TD error**:

$$
\delta_t \doteq R_{t+1} + \gamma V(S_{t+1}) - V(S_t).
\tag{6.5}
$$

It is the error in the estimate made at time $t$, not an error available at time $t$: it depends on $R_{t+1}$ and $S_{t+1}$ and therefore becomes available only at time $t+1$.

If $V$ does not change during the episode, as in Monte Carlo methods, the Monte Carlo error is the sum of the discounted TD errors. Starting from the recursive definition of the return,

$$
\begin{aligned}
G_t - V(S_t)
&= R_{t+1} + \gamma G_{t+1} - V(S_t) \\
&= \delta_t + \gamma\left[G_{t+1} - V(S_{t+1})\right] \\
&= \delta_t + \gamma\delta_{t+1} + \gamma^2\delta_{t+2} + \cdots + \gamma^{T-t-1}\delta_{T-1} \\
&= \sum_{k=t}^{T-1} \gamma^{k-t}\delta_k.
\end{aligned}
\tag{6.6}
$$

This identity is exact when $V$ is not updated during the episode; when $V$ is updated during the episode but the step size is small, it may still hold approximately.

## Advantages of TD Prediction Methods

TD methods update estimates partly from other estimates, learning a guess from a guess. Despite this, they have several advantages over DP and Monte Carlo methods.

### Practical Advantages

- Unlike DP methods, TD methods do not require a model of the environment, its reward distributions, or its next-state probability distributions.
- Unlike Monte Carlo methods, TD methods are naturally online and fully incremental. Monte Carlo methods must wait until the end of an episode because the return is not known earlier, whereas TD methods wait only one time step.
- TD methods can learn from nonterminating episodes and continuing tasks, and can be used when episodes are very long and delaying all learning until their end would be too slow.
- TD methods are less susceptible than Monte Carlo methods to problems caused by actions that produce very long transitions.

### Convergence

For any fixed policy $\pi$, tabular TD(0) converges in the mean to $v_\pi$ when the constant step-size parameter is sufficiently small. If the step-size parameter decreases according to the usual stochastic approximation conditions, it converges with probability 1.

Most convergence proofs apply only to the table-based case of the algorithm presented above. Some also apply to the case of general linear function approximation.

### Learning Speed

Both TD and Monte Carlo methods converge asymptotically to the correct predictions, but this does not determine which method reaches them faster. No mathematical proof establishes that one method always converges faster than the other. In practice, TD methods have usually converged faster than constant-$\alpha$ Monte Carlo methods on stochastic tasks.

## Optimality of TD(0)

### Batch Updating

When only a finite amount of experience is available, **batch updating** repeatedly presents all available experience to the learning method until the value function converges. Updates are made only after processing the complete batch of training data. Under batch updating, TD(0) converges deterministically to a single answer independent of the step-size parameter $\alpha$, provided $\alpha$ is sufficiently small. Constant-$\alpha$ Monte Carlo methods also converge deterministically under the same conditions, but generally to a different answer.

### Example 6.3: Random Walk Under Batch Updating

Batch versions of TD(0) and constant-$\alpha$ Monte Carlo were applied to the random-walk prediction example with $\alpha = 0.05$. After each new episode, all episodes observed so far were repeatedly presented until the value function converged. The resulting value function was compared with $v_\pi$ using the root mean square error over the five states, averaged over 100 independent repetitions. Batch TD was consistently better than batch Monte Carlo.

Under batch training, constant-$\alpha$ Monte Carlo converges to values $V(s)$ that are sample averages of the actual returns following visits to each state $s$. These are optimal estimates in the sense that they minimize the mean square error from the actual returns in the training set. Batch TD(0), however, finds estimates that are better according to the root mean square error from $v_\pi$.

### Example 6.4: You Are the Predictor

Suppose the following eight episodes are observed:

```text
A, 0, B, 0    B, 1
B, 1          B, 1
B, 1          B, 1
B, 1          B, 0
```

The first episode starts in state $A$, transitions to $B$ with reward $0$, and then terminates from $B$ with reward $0$. The other seven episodes start in $B$ and terminate immediately; six produce return $1$ and one produces return $0$.

There are two reasonable answers for the optimal value of $V(A)$:

1. Every observed transition from $A$ goes immediately to $B$ with reward $0$. Because six of the eight observed transitions from $B$ produce reward $1$, $B$ has value $\frac{3}{4}$. Thus $A$ also has value $\frac{3}{4}$. This is the answer given by batch TD(0).
2. State $A$ was observed once, and the return following it was $0$. Thus $V(A)=0$. This is the answer given by batch Monte Carlo.

The first answer is expected to produce lower error on future data if the process is Markov. The second answer is better on the existing data.

### Certainty-Equivalence Estimate

Batch Monte Carlo methods find estimates that minimize mean square error on the training set. Batch TD(0) always finds the estimate that would be exactly correct for the maximum-likelihood model of the Markov process. This is the **certainty-equivalence estimate**.

For a finite Markov process, the maximum-likelihood model estimates the transition probability from state $i$ to state $j$ as

$$
\frac{\text{number of observed transitions from } i \text{ to } j}
{\text{number of observed transitions out of } i},
$$

and estimates the expected reward for that transition as the average of its observed rewards. The certainty-equivalence estimate is the value function that would be exactly correct if this model were the true process. Batch TD(0) converges to this estimate.

This helps explain why TD methods converge more quickly than Monte Carlo methods in the batch setting: TD methods converge to the certainty-equivalence estimate, whereas Monte Carlo methods do not. Although nonbatch methods do not achieve either the certainty-equivalence or minimum-square-error estimate exactly, they can be understood as moving roughly toward them. Nonbatch TD(0) may be faster than constant-$\alpha$ Monte Carlo because it moves toward the better estimate.

For $n$ states, computing the maximum-likelihood model may require $n^2$ memory, and computing the corresponding value function conventionally requires on the order of $n^3$ computational steps. TD methods approximate the same solution using no more than order $n$ memory and repeated computations over the training set. On tasks with large state spaces, this may be the only feasible way to approximate the certainty-equivalence solution.

## Sarsa: On-policy TD Control

TD prediction methods can be used for the control problem by following generalized policy iteration (GPI), using TD methods for the evaluation or prediction part. As with Monte Carlo methods, this requires a tradeoff between exploration and exploitation and again gives two main classes: on-policy and off-policy. Sarsa is an on-policy TD control method.

For an on-policy method, the first step is to learn an action-value function rather than a state-value function. In particular, the method estimates $q_\pi(s,a)$ for the current behavior policy $\pi$ and for all states $s$ and actions $a$. Then the policy can be changed toward greediness with respect to the current action-value estimate.

An episode consists of an alternating sequence of states and state-action pairs. The transition is from state-action pair to state-action pair, and the values of state-action pairs are learned:

$$
Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha\left[R_{t+1} + \gamma Q(S_{t+1},A_{t+1}) - Q(S_t,A_t)\right].
\tag{6.7}
$$

This update is done after every transition from a nonterminal state $S_t$. If $S_{t+1}$ is terminal, then $Q(S_{t+1}, A_{t+1})$ is defined as zero. The rule uses every element of the quintuple $(S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1})$, which gives rise to the name Sarsa.

### Sarsa Control

Sarsa is an on-policy control algorithm based on this prediction method. It estimates $q_\pi$ for the behavior policy $\pi$ and at the same time changes $\pi$ toward greediness with respect to $q_\pi$.

**Algorithm parameters:** step size $\alpha \in (0,1]$, small $\varepsilon > 0$  
**Initialize:** $Q(s,a)$ for all $s \in \mathcal{S}^+$ and $a \in \mathcal{A}(s)$ arbitrarily, except $Q(\text{terminal}, \cdot)=0$

```text
Loop for each episode:
    Initialize S
    Choose A from S using policy derived from Q (e.g., epsilon-greedy)
    Loop for each step of episode:
        Take action A, observe R, S'
        Choose A' from S' using policy derived from Q (e.g., epsilon-greedy)
        Q(S, A) <- Q(S, A) + alpha[R + gamma Q(S', A') - Q(S, A)]
        S <- S'; A <- A'
    until S is terminal
```

### Convergence

The convergence properties of Sarsa depend on the nature of the policy's dependence on $Q$. For example, one could use $\varepsilon$-greedy or $\varepsilon$-soft policies. Sarsa converges with probability 1 to an optimal policy and action-value function if all state-action pairs are visited an infinite number of times and the policy converges in the limit to the greedy policy. This can be arranged with $\varepsilon$-greedy policies by setting $\varepsilon = 1/t$.

## Q-learning: Off-policy TD Control

Q-learning is an off-policy TD control algorithm. Its update is

$$
Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha\left[R_{t+1} + \gamma \max_a Q(S_{t+1},a) - Q(S_t,A_t)\right].
\tag{6.8}
$$

The learned action-value function $Q$ directly approximates $q_*$, the optimal action-value function, independently of the policy being followed. The policy still matters because it determines which state-action pairs are visited and updated.

For correct convergence, all state-action pairs must continue to be updated. This is a minimal requirement for any method guaranteed to find optimal behavior in the general case. Under this assumption and a variant of the usual stochastic approximation conditions on the step-size parameters, $Q$ has been shown to converge with probability 1 to $q_*$.

### Q-learning Control

**Algorithm parameters:** step size $\alpha \in (0,1]$, small $\varepsilon > 0$  
**Initialize:** $Q(s,a)$ for all $s \in \mathcal{S}^+$ and $a \in \mathcal{A}(s)$ arbitrarily, except $Q(\text{terminal}, \cdot)=0$

```text
Loop for each episode:
    Initialize S
    Loop for each step of episode:
        Choose A from S using policy derived from Q (e.g., epsilon-greedy)
        Take action A, observe R, S'
        Q(S, A) <- Q(S, A) + alpha[R + gamma max_a Q(S', a) - Q(S, A)]
        S <- S'
    until S is terminal
```

### Backup

The Q-learning update updates a state-action pair, so the top node of the backup diagram is a small, filled action node. The backup is also from action nodes: the update maximizes over all possible actions in the next state. These bottom nodes form the leaves of the backup diagram. The arcs include the maximum over the next-state actions, so the arc across the chosen action is crossed.

### Example 6.6: Cliff Walking

The cliff-walking task compares Sarsa and Q-learning. The undiscounted episodic task has start and goal states, four movement actions, and a cliff region. Stepping into the cliff gives reward $-100$ and sends the agent immediately back to the start; all other transitions give reward $-1$.

With $\varepsilon$-greedy action selection, Q-learning learns the values of the optimal policy: the agent travels along the edge of the cliff. This can cause occasional falling off the cliff because of exploratory actions. Sarsa accounts for the action selection and learns the longer but safer path through the upper part of the grid. Although Q-learning learns the values of the optimal policy, its online performance is worse than Sarsa, which learns the roundabout policy. If $\varepsilon$ were gradually reduced, both methods would asymptotically converge to the optimal policy.

## Expected Sarsa

Expected Sarsa is like Q-learning except that, instead of taking the maximum over next state-action pairs, it uses the expected value under the current policy. It updates

$$
\begin{aligned}
Q(S_t,A_t)
&\leftarrow Q(S_t,A_t) + \alpha\left[R_{t+1} + \gamma \mathbb{E}_\pi[Q(S_{t+1},A_{t+1}) \mid S_{t+1}] - Q(S_t,A_t)\right] \\
&= Q(S_t,A_t) + \alpha\left[R_{t+1} + \gamma \sum_a \pi(a \mid S_{t+1})Q(S_{t+1},a) - Q(S_t,A_t)\right].
\end{aligned}
\tag{6.9}
$$

Given the next state $S_{t+1}$, this algorithm moves deterministically in the same direction as Sarsa moves in expectation. It is therefore called **Expected Sarsa**. Its backup differs from Q-learning by averaging over the possible next actions under the policy rather than maximizing over them.

Expected Sarsa is more complex computationally than Sarsa, but it removes variance due to the random selection of $A_{t+1}$. With the same amount of experience, it can be expected to perform slightly better than Sarsa; in the cliff-walking task it performed substantially better than Sarsa and Q-learning.

In the cliff-walking task, Expected Sarsa retains Sarsa's advantage over Q-learning over a wide range of step-size values. Sarsa performs well in the long run at small $\alpha$, where short-term performance is poor. Q-learning can perform worse than Sarsa on asymptotic performance, while Expected Sarsa shows significant improvement over Sarsa for both interim and asymptotic performance.

In the cliff-walking results, Expected Sarsa was used on-policy, but it can generally be used either on-policy or off-policy. For example, if the target policy is the greedy policy while behavior is more exploratory, then Expected Sarsa is exactly Q-learning. It subsumes and generalizes Q-learning, and may improve over Sarsa except for the additional computational cost.

## Maximization Bias and Double Learning

Control algorithms such as Q-learning construct target policies with maximization over estimated action values. This can create a positive **maximization bias**: even if all true values are zero, uncertain estimates may produce a positive maximum.

### Example 6.7: Maximization Bias

In the small MDP, there are two nonterminal states, $A$ and $B$. From $A$, the agent can go left or right. The right action transitions immediately to the terminal state with reward $0$. The left action transitions to $B$ with reward $0$. From $B$, there are many actions; all transition immediately to the terminal state. Their rewards are normally distributed with mean $-0.1$ and variance $1.0$. Therefore, any trajectory starting with left from $A$ has expected return $-0.1$, and taking right in $A$ is always a mistake.

Because Q-learning uses the maximum of estimated values, it can initially learn to take the left action much more often than the right action. Double Q-learning is essentially unaffected by this maximization bias. Its value estimates were twice as large as Q-learning's because two separate action-value estimates were learned, but its learned policy used $Q_1 + Q_2$.

### Double Learning

The bias can be addressed by splitting experience into two sets and learning two independent estimates, $Q_1(a)$ and $Q_2(a)$. One estimate determines the maximizing action, and the other estimates its value. For example, if $Q_1$ determines

$$
A^* = \arg\max_a Q_1(a),
$$

then $Q_2(A^*)$ is an unbiased estimate of the value of that action. Symmetrically, $Q_2$ can determine the action and $Q_1$ can estimate its value. This is the idea of **double learning**. Although two estimates are learned, only one is updated on each play, so the computational cost per step is not doubled.

### Double Q-learning

Double Q-learning divides time steps in two, perhaps by flipping a coin on each step. If the coin comes up heads, the update is

$$
Q_1(S_t,A_t) \leftarrow Q_1(S_t,A_t) + \alpha\left[R_{t+1} + \gamma Q_2\left(S_{t+1}, \arg\max_a Q_1(S_{t+1},a)\right) - Q_1(S_t,A_t)\right].
\tag{6.10}
$$

If the coin comes up tails, the same update is done with $Q_1$ and $Q_2$ switched. The two approximate value functions are treated symmetrically. The behavior policy can use both estimates; for example, an $\varepsilon$-greedy policy for Double Q-learning can be based on the average or sum of the two action-value estimates.

### Double Q-learning Algorithm

**Algorithm parameters:** step size $\alpha \in (0,1]$, small $\varepsilon > 0$  
**Initialize:** $Q_1(s,a)$ and $Q_2(s,a)$ for all $s \in \mathcal{S}^+$ and $a \in \mathcal{A}(s)$ such that $Q(\text{terminal}, \cdot)=0$

```text
Loop for each episode:
    Initialize S
    Loop for each step of episode:
        Choose A from S using the policy epsilon-greedy in Q1 + Q2
        Take action A, observe R, S'
        With 0.5 probability:
            Q1(S, A) <- Q1(S, A) + alpha[R + gamma Q2(S', argmax_a Q1(S', a)) - Q1(S, A)]
        else:
            Q2(S, A) <- Q2(S, A) + alpha[R + gamma Q1(S', argmax_a Q2(S', a)) - Q2(S, A)]
        S <- S'
    until S is terminal
```

Double learning can be combined with Sarsa and Expected Sarsa as well. The double learning idea is not limited to Q-learning.

## Games, Afterstates, and Other Special Cases

A conventional action-value function assigns values to state-action pairs. For some tasks, it is more efficient to learn a state-value function over the positions reached immediately after the agent acts. These positions are called **afterstates**, and their value functions are **afterstate value functions**.

Afterstates are useful when the immediate effects of the agent's action are known, but the full dynamics are not. In games, for example, the rules usually specify the move that results from an action, while the opponent's reply is unknown. Afterstate value functions use this known structure and can produce a more efficient learning method.

In tic-tac-toe, different actions from different positions can produce the same resulting position. A conventional action-value function would have to learn these action values separately, but an afterstate value function immediately assesses both pairs equally because it evaluates the common resulting position. Any learning about that position is transferred immediately to all state-action pairs that produce it.

Afterstates arise in many tasks, including queuing tasks where actions assign customers to servers, reject customers, or discard information. The exact effects of the actions are defined, but the later dynamics may be complex.

These special cases do not require a different overall principle. They are usually still handled by generalized policy iteration, with a policy and an afterstate value function interacting in essentially the same way. In many cases there remains a choice between on-policy and off-policy methods for managing persistent exploration.